# Deep RecSys Course
## Домашнее задание 1

### ФИО: Швецов Олег Андреевич

### Введение
В этом домашнем задании вы полностью пройдёте базовый пайплайн: подготовка данных → метрики → несколько рекомендательных подходов → итоговый лидерборд.

### Используемые библиотеки
В данном задании потребуются следующие библиотеки:
* [polars](https://pola.rs/) - библиотека для работы с данными (человечество постепенно уходит от `pandas`'а)
* [implicit](https://github.com/benfred/implicit) - библиотека для обучения и применения различных коллаборативных рекомендательных моделей
* [torch](https://pytorch.org/) - no comments
* [gensim](https://radimrehurek.com/gensim/) - обучение **word2vec**

### Данные
Данные лежат в архиве `data.zip`, который состоит из:
* `interactions.parquet` - user-item взаимодействия из датасета Yambda (лайки для 500m версии)
* `embeddings.parquet` - уже пофильтрованные и чуть более плотно запакованные эмбеддинги треков из Yambda
* `artists.parquet` - метаданные айтемов с маппингом в артистов

Скачать архив можно здесь: [ссылка на google disk](https://drive.google.com/file/d/1PojPVpXGBAqzHQi97QAFhJ9gnPsXxveS/view?usp=sharing). В следующем блоке мы в любом случае скачиваем датасет, поэтому самостоятельно его можно не качать.

### Guidelines
- Для выполнения ДЗ достаточно использовать Google Collab с T4
- Детерминизм: фиксируйте сиды там, где это важно
- Не используйте данные из теста при обучении/подготовке моделей
- Тест — **последняя неделя** (по timestamp), как описано ниже
- После каждого этапа запускайте проверки внутри ноутбука
- Старайтесь избегать работы с сырыми питоновскими объектами (словарями, списками, интами) там, где можно применить методы из `polars` - они будут в десятки-сотни раз быстрее и читабельней
- Чтобы быстрее обнаруживать, что код написан неоптимально и выполняется слишком долго, старайтесь оборачивать циклы в `tqdm` и выводить progress bar
- Во время отладки кода можно посэмплировать данные для скорости с помощью `data.sample(fraction=0.1, seed=42)`. Для проверки тестов после отладки надо сделать запуски с полным датасетом

### Разбалловка
1. Подготовка данных (1 балл)
2. Оценка качества (1 балл)
3. Топ популярного (1 балл)
4. Рекомендации по артистам (1 балл)
5. Item-to-Item рекомендации (2 балла)
6. Item2Vec (1 балл)
7. Item-based Collaborative Filtering (1 балл)
8. ALS (1 балл)
9. Вопросы (1 балл)

Суммарно - 10 баллов.

In [1]:
from sympy.printing.pytorch import torch
! python --version

Python 3.12.4


In [3]:
import tests

!pip install gensim
# implicit устанавливается долго, минут 10
# !pip install implicit

!pip install -q gdown
!gdown --id 1PojPVpXGBAqzHQi97QAFhJ9gnPsXxveS -O dataset.zip
!unzip -q dataset.zip

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1PojPVpXGBAqzHQi97QAFhJ9gnPsXxveS
From (redirected): https://drive.google.com/uc?id=1PojPVpXGBAqzHQi97QAFhJ9gnPsXxveS&confirm=t&uuid=d024e934-4d41-4ad6-9a39-a89d3919834f
To: /content/dataset.zip
100% 356M/356M [00:06<00:00, 51.1MB/s]
replace artists.parquet? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace embeddings.parquet? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace interactions.parquet? [y]es, [n]o, [A]ll, [N]one, [r]ename: n


### 1. Подготовка данных (1 балл)

**Задача:**
1) Считать данные (взаимодействия, эмбеддинги, метаданные).  
2) Оставить только взаимодействия, для которых есть эмбеддинги.  
3) Поджоинить метаданные (артисты треков) ко всем взаимодействиям.  
4) Сделать core фильтрацию: оставить только айтемы с ≥5 взаимодействиями (в ДЗ мы делаем это исключительно для удобства и скорости, в реальной работе так делать не стоит)
5) Сделать train-test split: последнюю неделю положить в тест.  
6) Ограничить тест юзерами, у которых есть взаимодействия в трейне.  
7) Подготовить для оценки качества `test_targets: Dict[uid, List[item_id]]`.

После этого блока должны существовать `train`, `test`, `embeddings`, `artists`, `test_targets`.

Для этого блока полезны как минимум следующие методы:
* `pl.read_parquet` - для чтения данных
* `.filter, .value_counts` помогут сделать core-фильтрацию
* `df.join(...)` - при джойне метаданных надо использовать `how='left'`, а не `how='inner'`
* `df.join(other, on=some_key, how='semi')` - режим `semi` используется для фильтраций (оставить только те строки из исходного датафрейма, ключ которых присутствует во второй таблице)

In [2]:
import tests

In [3]:
from typing import Dict, List, Tuple, Any, Optional
from collections import defaultdict
import os

import numpy as np
import polars as pl

import tests

# Пути к данным (ожидается, что они лежат рядом с ноутбуком)
DATA_DIR = "."
PATH_INTERACTIONS = os.path.join(DATA_DIR, "interactions.parquet")
PATH_EMBEDDINGS = os.path.join(DATA_DIR, "embeddings.parquet")
PATH_ARTISTS = os.path.join(DATA_DIR, "artists.parquet")

# Глобальные параметры
TOPK = 100
CORE_MIN_INTERACTIONS_PER_ITEM = 5
TEST_INTERVAL_SECONDS = 7 * 24 * 60 * 60

# Для воспроизводимости
np.random.seed(42)

data = pl.read_parquet(PATH_INTERACTIONS)
embeddings = pl.read_parquet(PATH_EMBEDDINGS)
artists = pl.read_parquet(PATH_ARTISTS)

In [4]:
embeddings_items = embeddings.select(pl.col("item_id").unique())
data = data.join(embeddings_items, on="item_id", how="semi")

In [5]:
item_counts = data["item_id"].value_counts()

In [6]:
popular_items = item_counts.filter(pl.col("count") >= CORE_MIN_INTERACTIONS_PER_ITEM).select("item_id")

In [7]:
data = data.join(popular_items, on="item_id", how="semi")

In [8]:
item_counts = data.group_by("item_id").len()
valid_item_ids = item_counts.filter(pl.col("len").ge(5))
data = data.filter(pl.col("item_id").is_in(valid_item_ids["item_id"].to_list()))

In [9]:
data = data.join(artists, on="item_id", how="left")

In [10]:
max_ts = data["timestamp"].max()
test_start_ts = max_ts - TEST_INTERVAL_SECONDS
train = data.filter(pl.col("timestamp") < test_start_ts)
test = data.filter(pl.col("timestamp") >= test_start_ts)

In [11]:
train_users = train.select(pl.col("uid").unique())
test = test.join(train_users, on="uid", how="semi")

In [12]:
test_targets = dict(
    test.group_by("uid")
    .agg(pl.col("item_id"))
    .iter_rows()
)

In [13]:
# Запуск автопроверок (для них необходим файл tests.py)

tests.check_data_split(train=train, test=test, embeddings=embeddings, artists=artists, test_targets=test_targets)

All good! :)


### 2. Оценка качества (1 балл)

#### 2.1 Определения метрик

2.1.1 Пусть для пользователя $u$:

* $G_u \subset \mathcal{I}$ — множество релевантных айтемов (ground truth)
* $R_u = (r_{u,1}, \dots, r_{u,K})$ — упорядоченный список рекомендаций длины $K$

Обозначим индикатор релевантности $I_{u,k} = [ r_{u, k} \in G_u]$. В простонародье его еще часто называют `hits`.

2.1.2 **Hitrate@K** равен единичке, если мы угадали в topK хотя бы один релевантный айтем:
* $
\text{Hitrate@K} = \frac{1}{|U|}
\sum_{u \in U}
\left[ \sum_{k=1}^{K} I_{u,k} > 0 \right]
$

2.1.3 **Recall@K** оценивает долю угаданных релевантных айтемов (от всех релевантных айтемов):
* $
\text{Recall@K} = \frac{1}{|U|}
\sum_{u \in U}
\frac{
\sum_{k=1}^{K} I_{u,k}
}{
\min(|G_u|, K)
}
$

2.1.4 Для подсчета **nDCG@K** нужно сначала посчитать **DCG@K**, затем посчитать **iDCG@K** (DCG в случае идеального ранжирования), затем одно поделить на другое:
* $
\text{DCG@K}(u) = \sum_{k=1}^{K}
\frac{I_{u,k}}{\log_2(k+1)}
$
* $
\text{iDCG@K}(u) = \sum_{k=1}^{\min(|G_u|,K)}
\frac{1}{\log_2(k+1)}
$
* $
\text{nDCG@K} = \frac{1}{|U|}
\sum_{u \in U}
\frac{\text{DCG@K}(u)}{\text{iDCG@K}(u)}
$

2.1.5 **Coverage@K** - это число уникальных айтемов во всех рекомендациях, деленное на размер каталога:

* $
\text{Coverage@K} = \frac{|\bigcup_{u \in U} R_u|}{|\mathcal{I}_{train}|},
$ где $\mathcal{I}_{train}$ — каталог айтемов в train.
* в качестве размера каталога используем количество айтемов, которые нам доступны для рекомендации на момент рекомендации (то есть количество уникальных айтемов в `train`)

#### 2.2 Что нужно сделать
Реализуйте функции:
- `get_metrics(targets, candidates, topk) -> dict(hitrate, recall, ndcg)`
- `evaluate(targets_by_user, candidates_by_user, catalog_size, topk) -> dict(hitrate, recall, ndcg, coverage)`

**Важно:** `candidates[uid]` должен иметь длину ровно `topk`.  


In [14]:
from tqdm import tqdm

In [15]:
def get_metrics(targets: List[int], candidates: List[int], topk: int) -> Dict[str, float]:
    def I_k(cand) -> int:
        return int(cand in targets)
    hitrate = int(sum([I_k(cand) for cand in candidates]) > 0)
    recall = sum([I_k(cand) for cand in candidates]) / min(topk, len(targets))
    dcg = sum([I_k(candidates[k-1])/np.log2(k+1) for k in range(1, topk+1)])
    idcg = sum([1/np.log2(k+1) for k in range(1, min(topk+1, len(targets)+1))])
    ndcg = dcg/idcg
    return {
        "hitrate": hitrate,
        "recall": recall,
        "ndcg": ndcg,
    }


def evaluate(
    targets: Dict[int, List[int]],
    candidates: Dict[int, List[int]],
    catalog_size: int,
    topk: int = 100,
) -> Dict[str, float]:
    users_metrics = []
    for uid in targets:
        user_target = targets[uid]
        user_candidates = candidates[uid]
        users_metrics.append(get_metrics(user_target, user_candidates, topk))


    hitrate = np.mean([metrics['hitrate'] for metrics in users_metrics])
    recall = np.mean([metrics['recall'] for metrics in users_metrics])
    ndcg = np.mean([metrics['ndcg'] for metrics in users_metrics])

    candidates_set = []
    for cands in candidates.values():
        candidates_set.extend(cands)
    coverage = len(set(candidates_set)) / catalog_size
    return {
        "hitrate": hitrate,
        "recall": recall,
        "ndcg": ndcg,
        "coverage": coverage,
    }

In [16]:
tests.check_metrics(get_metrics=get_metrics, evaluate=evaluate)

All good! :)


### 3. Топ популярного (1 балл)

Сделайте топ популярных айтемов по train взаимодействиям и посчитайте метрики с помощью `evaluate`.

Полезные методы из `polars`: `.value_counts, .sort, .head, .to_numpy, .tolist, .n_unique`.

**Важно:** в этом пункте не нужно фильтровать для каждого пользователя айтемы, которые уже были в истории пользователя. Нужно просто сделать для всех пользователей один и тот же список кандидатов.

In [17]:
item_counts = train["item_id"].value_counts()
popular_items = (
    item_counts
    .sort("count", descending=True)
    .head(TOPK)
    ["item_id"]
    .to_list()
)

In [18]:
candidates_by_user = {uid: popular_items for uid in test_targets.keys()}
catalog_size = train["item_id"].n_unique()
metrics_toppop = evaluate(test_targets, candidates_by_user, catalog_size, TOPK)

In [19]:
# нужно сложить результат метода evaluate в словарь metrics_toppop,
# далее в ноутбуке аналогично ожидаются правильные названия словарей с метриками
tests.check_top_pop(metrics_toppop)

All good! :)


**Важно!** Проверка выше требовала точное совпадение метрик с эталонным ноутбуком, чтобы убедиться в корректности логики обработки данных.
Везде дальше мы требуем метрики выше определенного порога, а не точное совпадение. При этом порог взят с запасом - выбраны ощутимо меньшие значения, чем реальные значения в эталонном ноутбуке. Удачи! :)

### 4. Рекомендации по артистам (1 балл)

Что хотим:
1. Взять последние N лайков пользователя (для моделей и всех подсчетов нужно использовать ТОЛЬКО `train`, тест используется исключительно для оценки качества)
2. Посчитать по ним любимых артистов пользователя (отсортировать по количеству лайков у артиста)
3. Взять у любимых артистов их самые популярные треки (чтобы посчитать популярность треков, нужно использовать train)
4. Оставить те, которые пользователь еще не видел (не лайкал)
5. Их порекомендовать

Фактически, мы формируем рекомендации по счётчикам - учитываем "сколько раз пользователь лайкал артиста" и "сколько раз трек артиста был лайкнут".

В этом методе 100 кандидатов найдётся далеко не для всех пользователей, поэтому нам нужно дополнить рекомендации до 100 каким-то дополнительным методом; то есть сделать fallback. В данном случае предлагается делать fallback на `top_pop` рекомендации:
* если у пользователя набралось меньше 100 рекомендаций, то идём по top_pop и добавляем кандидатов в рекомендации, пока не заполним до 100
* при этом добавляем только треки, которых не было в уже отобранных до fallback'а рекомендациях

In [20]:
# именно такие значения ожидает автопроверка tests.check_artist_recs
N_LAST_EVENTS = 100  # n последних взаимодействий
PER_ARTIST_LIMIT = 20  # берем для рекомендаций 20 айтемов из каждого любимого артиста

In [21]:
last_likes_per_uid = (
    train
    .sort(by='timestamp', descending=True)
    .group_by('uid', maintain_order=True)
    .agg(
        pl.col('artist_id')
        .head(N_LAST_EVENTS)
        .alias('artist_ids')
    )
    .select(['uid', 'artist_ids'])
    .to_dict(as_series=False)
)

last_likes_per_uid = dict(zip(
    last_likes_per_uid['uid'],
    last_likes_per_uid['artist_ids']
))

In [22]:
uids = train.select(pl.col('uid').unique())['uid'].to_list()

In [23]:
artist_pop_items = (
    train
    .group_by(['artist_id', 'item_id'])
    .len()
    .sort(by=['artist_id', 'len'], descending=[False, True])
    .group_by('artist_id', maintain_order=True)
    .agg(pl.col('item_id'))
    .to_dict(as_series=False)
)

artist_pop_items = dict(zip(
    artist_pop_items['artist_id'],
    artist_pop_items['item_id']
))

In [24]:
artist_ids = train.select(pl.col('artist_id').unique())['artist_id'].to_list()

In [25]:
listened = (
    train
    .unique(subset=['uid', 'item_id'])
    .group_by('uid')
    .agg(pl.col('item_id'))
    .to_dict(as_series=False)
)

listened = dict(zip(
    listened['uid'],
    [set(items) for items in listened['item_id']]
))

In [26]:
most_pop_items_uid = dict()

for uid in tqdm(uids):
    artists_ids = last_likes_per_uid[uid]
    count_dict = dict()
    for art_id in artists_ids:
        if art_id not in count_dict:
            count_dict[art_id] = 1
        else:
            count_dict[art_id] += 1
    artist_list = [(art_id, art_count) for art_id, art_count in count_dict.items()]
    sorted_artists = sorted(artist_list, key=lambda x: x[1], reverse=True)

    top_pop_items = []
    for art_id, _ in sorted_artists:
        pop_items = artist_pop_items[art_id][:PER_ARTIST_LIMIT]
        filtered_items = []
        for item in pop_items:
            if item not in listened[uid]:
                filtered_items.append(item)
        top_pop_items.extend(filtered_items[:PER_ARTIST_LIMIT])
    most_pop_items_uid[uid] = top_pop_items


100%|██████████| 81020/81020 [00:04<00:00, 19142.87it/s]


In [27]:
def fallback_to_toppop(cands_by_uid: Dict[int, List[int]], top_pop: np.ndarray, topk: int) -> Dict[int, List[int]]:
    assert type(top_pop) == list
    for uid, cands in cands_by_uid.items():
        size = len(cands)
        if size < topk:
            set_cands = set(cands)
            additional_recs = []
            for item in top_pop: # перебираем самые популярные айтемы
                if item not in set_cands:
                    assert type(item) == int
                    additional_recs.append(item)
            cands = cands + additional_recs # добавляем и обрезаем до 100
            cands = cands[:topk]
        else:
            cands = cands[:topk]
        cands_by_uid[uid] = cands # обновляем кандидатов
    return cands_by_uid

candidates = most_pop_items_uid
candidates_by_user = fallback_to_toppop(candidates, popular_items, TOPK)


In [28]:
metrics_artist = evaluate(test_targets, candidates_by_user, catalog_size, N_LAST_EVENTS)
metrics_artist

{'hitrate': np.float64(0.22712706297067778),
 'recall': np.float64(0.06332134639251796),
 'ndcg': np.float64(0.02462620310137259),
 'coverage': 0.6315404340882048}

In [29]:
tests.check_artist_recs(metrics_artist)

All good! :)


### 5. Item-to-Item рекомендации (2 балла)

Здесь предлагается реализовать классический item-to-item алгоритм:
1. Считаем для каждого айтема список похожих айтемов
2. Берём последние N взаимодействий пользователя, для каждого вытаскиваем список похожих
3. Агрегируем кандидатов из списка похожих, учитывая суммарные похожести (если айтем встретился в нескольких списках кандидатов у пользователя, суммируем похожесть из каждого)
4. Отфильтровываем все, что уже было у пользователя в истории
5. Оставляем только topk кандидатов

Реализация должна состоять из двух функций:
1) `get_similar_items_gpu(item_ids, item_embeddings, topk_sim)` - для каждого айтема находим top similar items по **косинусу**
* используем pytorch (и GPU), иначе подсчет будет очень долгим
* создаем тензор с эмбеддингами, переводим на GPU, $l_2$-нормализуем эмбеддинги (`torch.nn.functional.normalize`)
* идём батчами по эмбеддингам, для каждого батча с помощью `torch.topk` считаем честный топ похожих; бачти - это важно! Батчевание сильно влияет на скорость
* запоминаем эту информацию в словаре вида `item_id -> [(cand_item_id, similarity), ...]`

2) `get_candidates_item2item(interactions, similar_items, ...)` - для каждого пользователя агрегируем похожести по последним N взаимодействиям.
* нужно буквально реализовать вышеописанный алгоритм "берём последние N взаимодействий, суммируем похожести всех полученных похожих айтемов"
* для написания этой функции лучше не пытаться сделать супер оптимальный код через `polars` (у него очень большое пиковое потребление оперативной памяти); в данном случае проще манипулировать словарями, списками, etc
* предлагается использовать `heapq.nlargest` для ускорения поиска `topk` элементов среди уже подсчитанных суммарных похожестей (раза в полтора-два быстрее, чем делать полную сортировку всех кандидатов со всех списков)

В пункте 2 нужно дополнительно поддержать **time decay** (затухание по времени):
* свежие взаимодействия должны вносить более высокий вклад, поэтому при суммировании похожестей для конкретного кандидата мы учитываем свежесть события, из списка которого берём похожесть
* вес события $2^{-\frac{L-t}{\tau}}$, где $L$ - длина истории пользователя, $t$ - позиция события в истории (начиная с единички), $\tau$ - период полураспада, то есть насколько быстро затухает сигнал от событий. Например, при $\tau = 10$ вес события падает в два раза, если оно идет на 10 позиций раньше в истории
* Получаем $\text{score}(u, j) =
\sum_{t=1}^{L}
2^{- \frac{L - t}{\tau}}
\cdot
s(i_{u,t}, j)
$

**Важно:** при использовании метода `get_similar_items_gpu` набираем не 100 кандидатов, а в два раза больше (2 * topk = 200), чтобы после фильтрации по истории пользователя и слияния всех списков у нас было больше шансов набрать `topk`.

Полезные методы из `polars`: `.to_torch()`, `.to_numpy()` - позволяют сразу получить тензоры для айдишников / эмбеддингов

In [30]:
valid_ids = item_counts.filter(pl.col("count") >= CORE_MIN_INTERACTIONS_PER_ITEM).select("item_id")

In [31]:
embeddings = embeddings.join(valid_ids, on="item_id", how="semi")

In [32]:
item_ids_ = embeddings['item_id'].to_numpy()
embeddings_series = embeddings['embed']
item_embeddings_ = np.vstack(embeddings_series.to_list())

In [34]:
uids = train.select(pl.col("uid").unique())['uid'].to_list()

In [86]:
import torch
import torch.nn.functional as F
import heapq

def get_similar_items_gpu(
    item_ids: np.ndarray,
    item_embeddings: np.ndarray,
    block: int = 1024,
    topk: int = 200,
    device: str = "cuda",
) -> Dict[int, List[Tuple[int, float]]]:
    embeddings_tensor = torch.FloatTensor(item_embeddings).to(device)
    embeddings_tensor = F.normalize(embeddings_tensor, p=2, dim=1)

    num_items = embeddings_tensor.shape[0]
    similar_items = {}

    k_needed = min(2 * topk, num_items)

    for start in tqdm(range(0, num_items, block)):
        end = min(start + block, num_items)
        batch = embeddings_tensor[start:end]

        similarities = torch.mm(batch, embeddings_tensor.T)

        top_similarities, top_indices = torch.topk(similarities, k=k_needed, dim=1)

        vals_np = top_similarities.cpu().numpy()
        idx_np = top_indices.cpu().numpy()

        batch_size = end - start
        current_indices = np.arange(start, end)

        for i in range(batch_size):
            item_idx = current_indices[i]
            item_id = int(item_ids[item_idx])

            mask = idx_np[i] != item_idx
            valid_idx = idx_np[i][mask][:topk]
            valid_sim = vals_np[i][mask][:topk]

            cand_ids = item_ids[valid_idx]

            item_similarities = list(zip(map(int, cand_ids), map(float, valid_sim)))
            similar_items[item_id] = item_similarities

    return similar_items


def get_candidates_item2item(
    interactions: pl.DataFrame,
    similar_items: Dict[int, List[Tuple[int, float]]],
    n_last: int = 30,
    half_life_frac: float = 0.5,
    topk: int = 100,
) -> Dict[int, List[int]]:
    """
    half_life_frac интерпретируется как доля n_last: half_life = half_life_frac * n_last.
    То есть в реальной формуле вам нужно домножить параметр half_life_frac на n_last
    w(r) = 2^{-r / half_life}, где r=0 для самого последнего события.
    """
    last = (
        interactions
        .sort('timestamp', descending=True)
        .group_by('uid')
        .agg([
            pl.col('item_id').head(n_last).alias('item_ids'),
            pl.col('timestamp').head(n_last).alias('timestamps')
        ])
    )
    history = {
        row['uid']: list(zip(row['item_ids'], row['timestamps']))
        for row in last.to_dicts()
    }

    half_life = half_life_frac * n_last
    candidates_i2i = {}

    for uid, user_history in tqdm(history.items()):
        scores = {}
        seen_items = {item_id for item_id, _ in user_history}

        for pos, (item_id, _) in enumerate(user_history):
            weight = 2 ** (-pos / half_life)

            if item_id not in similar_items:
                continue

            for cand_id, sim in similar_items[item_id]:
                if cand_id in seen_items:
                    continue
                scores[int(cand_id)] = scores.get(cand_id, 0.0) + sim * weight

        candidates_topk = [c for c, _ in heapq.nlargest(topk, scores.items(), key=lambda x: x[1])]
        candidates_i2i[uid] = candidates_topk

    return candidates_i2i

In [43]:
similar_items_gpu = get_similar_items_gpu(item_ids_, item_embeddings_, block=1024, topk=100, device='mps')

100%|██████████| 151/151 [01:12<00:00,  2.08it/s]


In [164]:
candidates_i2i = get_candidates_item2item(train, similar_items_gpu, topk=95)

100%|██████████| 81020/81020 [00:52<00:00, 1546.29it/s]


In [165]:
# сделаю фоллбек для тех у кого не хватило айтемов
candidates_i2i = fallback_to_toppop(candidates_i2i, popular_items, 100)

In [166]:
metrics_i2i = evaluate(test_targets, candidates_i2i, catalog_size, N_LAST_EVENTS)

In [167]:
metrics_i2i

{'hitrate': np.float64(0.1257277145756556),
 'recall': np.float64(0.030621379674632465),
 'ndcg': np.float64(0.01061827801001909),
 'coverage': 0.9570684092977086}

In [169]:
tests.check_i2i_recs(metrics_i2i)

All good! :)


### 6. Item2Vec (1 балл)

Item-to-item можно использовать с буквально любыми похожестями айтемов.

Теперь давайте попробуем сами обучить эмбеддинги айтемов, посчитать похожести и опять применить item-to-item.

Будем использовать подход Item2Vec - применим **word2vec** к последовательностям айтемов вместо последовательностей слов.

Для этого предлагается использовать `gensim`:
* нужно создать `corpus` из списка списков строк вида `[['1', '2'], ['3', '4', '5']]`, в котором строки - это буквально строки айдишников айтемов, а внутренние списки - это сгруппированные взаимодействия пользователя (хронологически отсортированные)
* Вызвать метод `gensim.models.Word2Vec`. Предлагаемые настройки `vector_size=100, window=5, min_count=1, sg=0, epochs=5`

После обучения, эмбеддинги можно достать с помощью `w2v.wv.vectors`, а соответствующие айдишники - с помощью `w2v.wv.index_to_key`.

Далее предлагается применить пайплайн из прошлого пункта: `get_similar_items_gpu` + `get_candidates_item2item`

In [50]:
last_interactions = (
    train
    .sort('timestamp', descending=True)
    .group_by('uid')
    .agg([
        pl.col('item_id').alias('item_ids'),
    ])
)
items_history = [list(map(str, row['item_ids'])) for row in last_interactions.to_dicts()]

In [51]:
from gensim.models import Word2Vec
model = Word2Vec(
    sentences=items_history,
    vector_size=100, window=5, min_count=1, sg=0, epochs=5, seed=1,
)

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


In [52]:
w2v_embeddings = model.wv.vectors
w2v_item_ids = np.array(model.wv.index_to_key, dtype=int)

In [54]:
similar_items_gpu_w2v = get_similar_items_gpu(
    w2v_item_ids, w2v_embeddings, block=1024, topk=200, device='mps'
)

100%|██████████| 154/154 [01:58<00:00,  1.30it/s]


In [55]:
candidates_w2v = get_candidates_item2item(train, similar_items_gpu_w2v)

100%|██████████| 81020/81020 [00:53<00:00, 1522.17it/s]


In [56]:
metrics_w2v = evaluate(test_targets, candidates_w2v, catalog_size, N_LAST_EVENTS)

In [57]:
metrics_w2v

{'hitrate': np.float64(0.18490626502163116),
 'recall': np.float64(0.04639548539325842),
 'ndcg': np.float64(0.017268359595738962),
 'coverage': 0.9161030052749798}

In [58]:
tests.check_w2v_recs(metrics_w2v)

### 7. Item-based Collaborative Filtering (1 балл)

Теперь попробуем применить тот же item-to-item подход, но используя в качестве векторов большие разреженные векторы из user-item матрицы. Для подсчета разреженных близостей наш GPU пайплайн не подходит (не умеет работать с разреженными данными), поэтому будем использовать библиотеку `implicit`

1. Сначала нужно собрать CSR user-item матрицу из наших `train` взаимодействий с помощью `scipy.sparse.csr_matrix`
* предлагается использовать формат вида `csr_matrix((ones, (user_ids, item_ids)), shape=(num_users, num_items))`
* чтобы просуммировать взаимодействия с одними и теми же айтемами можно использовать `.sum_duplicates`
* также понадобится от исходных айдишников (которые необязательно от 1 до `num_items / num_users`) перейти к компактным айдишникам, сделав маппинг `{old_item_id: new_item_id}`. Простой вариант -- сделать это с помощью словаря, более быстрый - использовать метод вида `train.select("uid").unique().with_row_index()` (аналогично для айтемов). И еще очень хак для ускорения - чтобы в таблице со старым `old_item_id` получить `new_item_id`, достаточно сделать джойн: `interactions.join(mapping_table, on='old_item_id', how='left')`, где название `old_item_id` зависит от имплементации (скорее всего это будет просто `item_id`)

2. Чтобы с помощью `implicit` получить списки похожих, нужно использовать комбинацию из `CosineRecommender.fit`, и `.similar_items`

3. Чтобы сформировать рекомендации, используем нашу функцию `get_candidates_item2item`

**Warning:** неаккуратно написанный код в этом пункте может переполнить оперативную память.

In [59]:
user_id_mapping = (
    train
    .select('uid')
    .unique()
    .sort('uid')
    .with_row_index('user_idx', offset=0)
)

item_id_mapping = (
    train
    .select('item_id')
    .unique()
    .sort('item_id')
    .with_row_index('item_idx', offset=0)
)

In [60]:
interactions_agg = (
    train
    .group_by(['uid', 'item_id'])
    .agg(pl.len().alias('count'))
    .join(user_id_mapping, on='uid', how='left')
    .join(item_id_mapping, on='item_id', how='left')
    .select(['user_idx', 'item_idx', 'count'])
)

In [65]:
counts = interactions_agg['count'].to_numpy()
user_indices = interactions_agg['user_idx'].to_numpy()
item_indices = interactions_agg['item_idx'].to_numpy()

In [78]:
item_idx_to_id = dict(zip(item_id_mapping['item_idx'], item_id_mapping['item_id']))
user_idx_to_id = dict(zip(user_id_mapping['user_idx'], user_id_mapping['uid']))

In [97]:
user_id_to_idx = {v: k for k, v in user_idx_to_id.items()}
item_id_to_idx = {v: k for k, v in item_idx_to_id.items()}

In [66]:
from scipy.sparse import csr_matrix

user_item_matrix = csr_matrix(
    (counts, (user_indices, item_indices)),
    shape=(len(user_id_mapping), len(item_id_mapping))
)

In [67]:
from implicit.nearest_neighbours import CosineRecommender, tfidf_weight

In [220]:
def run_cosine(X: csr_matrix, K=100, N_cand=100, topk=100, half_life_frac=0.5, n_last=30):
    cosine_model = CosineRecommender(K=K)
    cosine_model.fit(X)

    similarities_cosine = dict()
    for item_id in tqdm(item_id_mapping['item_idx'], desc="Eval для cosine модели"):
        cand_ids, similarities = cosine_model.similar_items(itemid=item_id, N=N_cand)
        similar_items = list(zip(cand_ids, similarities))
        similarities_cosine[int(item_id)] = similar_items

    similarities_cosine_converted = dict()
    to_id = lambda x: item_idx_to_id[x]
    for item_id in tqdm(item_id_mapping['item_idx'], desc="Замена индексов для cosine модели"):
        similar_items = similarities_cosine[item_id]
        similar_items_converted = []
        for similar_item in similar_items:
            similar_items_converted.append((to_id(similar_item[0]), similar_item[1]))
        similarities_cosine_converted[item_idx_to_id[item_id]] = similar_items_converted

    candidates_cf = get_candidates_item2item(train, similarities_cosine_converted, topk=topk, n_last=n_last, half_life_frac=half_life_frac)
    candidates_cf = fallback_to_toppop(candidates_cf, popular_items, 100)
    return evaluate(test_targets, candidates_cf, catalog_size, N_LAST_EVENTS)

In [221]:
metrics_cf = run_cosine(user_item_matrix)

/Users/oleg/miniconda3/lib/python3.12/site-packages/implicit/utils.py:164: ParameterWarning: Method expects CSR input, and was passed coo_matrix instead. Converting to CSR took 0.026021718978881836 seconds
  warnings.warn(


  0%|          | 0/157157 [00:00<?, ?it/s]

100%|██████████| 81020/81020 [00:40<00:00, 2000.43it/s]


In [222]:
metrics_cf

{'hitrate': np.float64(0.172648614004166),
 'recall': np.float64(0.05117930620397395),
 'ndcg': np.float64(0.02042018601983149),
 'coverage': 0.9256348746794607}

In [223]:
tests.check_cf_recs(metrics_cf)

All good! :)


##### 7.2 TF-IDF

А теперь щепотка магии - с помощью `implicit.nearest_neighbours.tfidf_weight` модифицируем user-item матрицу и получим более высокие метрики.

Все, что нужно - это применить этот метод к матрице и затем проделать все те же операции, что и раньше (с вычислением похожестей и т.д.)

In [227]:
X_tfidf = tfidf_weight(user_item_matrix)
metrics_cf_tfidf = run_cosine(X_tfidf, K=120, N_cand=100, topk=95, n_last=100, half_life_frac=0.1)

metrics_cf_tfidf

/Users/oleg/miniconda3/lib/python3.12/site-packages/implicit/utils.py:164: ParameterWarning: Method expects CSR input, and was passed coo_matrix instead. Converting to CSR took 0.03460097312927246 seconds
  warnings.warn(


  0%|          | 0/157157 [00:00<?, ?it/s]

100%|██████████| 81020/81020 [01:34<00:00, 860.15it/s] 


{'hitrate': np.float64(0.26024141430326336),
 'recall': np.float64(0.08096938591601832),
 'ndcg': np.float64(0.03192300343869304),
 'coverage': 0.945659436105296}

In [228]:
tests.check_tfidf_recs(metrics_cf_tfidf)

All good! :)


### 8. ALS (1 балл)

С помощью `implicit.als.AlternatingLeastSquares` над той же самой разреженной user-item матрицей можно обучить ALS, что вам и предлагается сделать.

Чтобы сформировать рекомендации, наша функция `get_candidates_item2item` не нужна - достаточно использовать метод `model.recommend`.

Нужно обучить ALS с двумя версиями user-item матриц - исходной и tfidf-модифицированной.

In [210]:
from implicit.als import AlternatingLeastSquares


def run_als(
        X: csr_matrix,
        factors: int = 100,
        reg: float = 0.5,
        alpha: float = 0.1,
        iters: int = 20,
        n_last: int = 40,
        N_cands: int = 100,
):

    als_model = AlternatingLeastSquares(
        factors=factors,
        iterations=iters,
        alpha=alpha,
        regularization=reg,
        random_state=42,
    )

    als_model.fit(X)

    last = (
        train
        .sort('timestamp', descending=True)
        .group_by('uid')
        .agg(pl.col('item_id').head(n_last).alias('item_ids'))
    )
    candidates_als = {}

    for row in tqdm(last.to_dicts(), desc="ALS"):
        uid = row['uid']
        user_idx = user_id_to_idx[uid]

        history_item_ids = row['item_ids']

        recommendations = als_model.recommend(
            userid=user_idx, N=N_cands,
            user_items=user_item_matrix[user_idx:user_idx+1],
        )

        candidates_als_uid = []
        for item_idx, score in zip(recommendations[0], recommendations[1]):
            item_id = item_idx_to_id[item_idx]
            if item_id not in history_item_ids:
                candidates_als_uid.append(item_id)
                if len(candidates_als_uid) == TOPK:
                    break

        candidates_als[uid] = candidates_als_uid
    candidates_als = fallback_to_toppop(candidates_als, popular_items, 100)
    return evaluate(test_targets, candidates_als, catalog_size, N_LAST_EVENTS)

In [ ]:
metrics_als_raw = run_als(
    user_item_matrix, n_last=50, N_cands=95,
    factors=150, alpha=15, reg=0.1, iters=30,
)

In [213]:
metrics_als_raw

{'hitrate': np.float64(0.2997382898039844),
 'recall': np.float64(0.09425826811635438),
 'ndcg': np.float64(0.0344728652927321),
 'coverage': 0.18813670406027094}

In [215]:
metrics_als_tfidf = run_als(
    X_tfidf, n_last=50, N_cands=95,
    factors=150, alpha=15, reg=0.1, iters=30,
)

/Users/oleg/miniconda3/lib/python3.12/site-packages/implicit/utils.py:164: ParameterWarning: Method expects CSR input, and was passed coo_matrix instead. Converting to CSR took 0.10060906410217285 seconds
  warnings.warn(


  0%|          | 0/30 [00:00<?, ?it/s]


ALS: 100%|██████████| 81020/81020 [05:33<00:00, 242.74it/s]


In [216]:
metrics_als_tfidf

{'hitrate': np.float64(0.2967206110131923),
 'recall': np.float64(0.09331446590671302),
 'ndcg': np.float64(0.03409121183976484),
 'coverage': 0.3593349325833402}

In [217]:
tests.check_als_recs(metrics_als_raw, metrics_als_tfidf)

All good! :)


### 9. Лидерборд и выводы

Собираем таблицу со всеми методами и метриками.

Добавьте 5–10 строк выводов к экспериментам: что работает лучше и почему.


In [229]:
leaderboard = pl.DataFrame([
    {"method": "TopPop", **metrics_toppop},
    {"method": "User-Artist", **metrics_artist},
    {"method": "Item2Item (dataset emb)", **metrics_i2i},
    {"method": "Item2Vec (w2v)", **metrics_w2v},
    {"method": "CF Cosine (raw)", **metrics_cf},
    {"method": "CF Cosine (tf-idf)", **metrics_cf_tfidf},
    {"method": "ALS (raw)", **metrics_als_raw},
    {"method": "ALS (tf-idf)", **metrics_als_tfidf},
])

leaderboard = leaderboard.sort(["recall", "ndcg"], descending=True)
leaderboard

method,hitrate,recall,ndcg,coverage
str,f64,f64,f64,f64
"""ALS (raw)""",0.299738,0.094258,0.034473,0.188137
"""ALS (tf-idf)""",0.296721,0.093314,0.034091,0.359335
"""CF Cosine (tf-idf)""",0.260241,0.080969,0.031923,0.945659
"""User-Artist""",0.227127,0.063321,0.024626,0.63154
"""CF Cosine (raw)""",0.172649,0.051179,0.02042,0.925635
"""Item2Vec (w2v)""",0.184906,0.046395,0.017268,0.916103
"""Item2Item (dataset emb)""",0.125728,0.030621,0.010618,0.957068
"""TopPop""",0.112375,0.030366,0.010998,0.000636


### Вопросы на понимание (1 балл)

1) Почему для кандидатов/метрик важно исключать айтемы, которые пользователь уже видел?  
2) Почему при джойне метаданных (да и вообще почти при любом джойне) нужно использовать how='left', а не how='inner'?
3) В рекомендациях по артистам (с помощью счётчиков) не для всех пользователей может найтись нужное количество кандидатов. Почему?
4) Почему tf-idf улучшает item-based CF? На саму функцию можно посмотреть через `tfidf_weight??`
5) В чем принципиальное отличие между item-to-item методом и методом, при котором мы получаем эмбеддинг пользователя, сложив эмбеддинги его последних взаимодействий, и затем ищем ближайшие эмбеддинги айтемов?
5) Почему ALS выиграл у чистого cosine-item2item?

Ответы - текстом

1) если включить пару user-item в рекомендации, что есть в train данных, то модель будет хуже предсказывать новое, но при этом метрики не упадут, так как мы фактически угадаем, что будет дальше, а значит метрика стала менее репрезентативной, что плохо

2) потому что inner джойн обрежет строки в левой таблице, если нужного ключа из них нет в правой, что может сильно снижать размер обучающих данных и их репрезентативность. Метаданные могут быть доступны не для всех, например, когда на яндексе рекомендуют заблокированный западный трек под видом ноунейм исполнителя, по которому даже перейти на карточку артиста нельзя, то есть у этого трека нет метаданных артиста. Если сделать inner, то такие треки обрежутся, а если left, то они останутся в обучающих данных

3) потому что мы не сделали фильтрацию на число лайков пользователя. Например, там есть пользователь, который на train данных послушал только два трека, а значит если посмотреть на топ популярных треков у этих артистов, то очень велик шанс набрать меньше 100, так как не у всех исполнителей есть много треков (50 и больше)

4) TF-IDF = (частота взаимодействия юзера с треком) * (обратная частота трека среди всех юзеров), так что этот метод как бы сглаживает данные перед обучением cf. Например, если трек был популярен у всех юзеров, тот же моргенштерн в 2020, то без tfidf этот подход будет забивать рекомендации очевидными вариантами, а в контексте того, что у нас еще temporal split вполне, может возникнуть ситуация, что популярность моргенштерна упадет и никому наши рекомендации не зайдут, что обрушит метрики. При использовании tfidf он сгладит популярности.

5) сумма эмбеддингов айтемов описывает как бы средний трек, что любит пользователь, а i2i дает для каждого трека свои списки похожих и потом уже суммирует эти похожести, а не общую характеристику каждого трека (это я так эмбеддинг назвал). Таким образом i2i очень хорош в плане coverage (что как раз видно в таблице), а сумма эмбеддингов будет хороша для поиска обобщений

6) потому что ALS попеременно раскладывает user-item матрицу в произведение двух и это позволяет уловить более обобщенные связи, чем это делает Cosine, который просто считает схожести